In [1]:
import cortex
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from scipy.stats import zscore

print(cortex.database.default_filestore)
print(cortex.options.usercfg)


/data/jiwoongpark/pycortex_store
/home/jlg/jiwoongpark/.config/pycortex/options.cfg


In [ ]:
# Volumetric data:
root = '/data/jiwoongpark/statedynamics_data/storylistening/ANfs/nifti_export/'
prediction_accuracy = nib.load(f'{root}/ANfs_static_R2.nii.gz').get_fdata()                 #3D  (68, 86, 36)        test R²
predicted_response = nib.load(f'{root}/ANfs_predicted_response_test.nii.gz').get_fdata()    #4D  (68, 86, 36, 291)   redicted response
actual_response = nib.load(f'{root}/ANfs_actual_response_test.nii.gz').get_fdata()          #4D  (68, 86, 36, 291)   actual BOLD response
design_matrix_1 = np.sin(np.linspace(0, 6*np.pi, 291))                                      #1D  (291,)              Design matrix 1
design_matrix_2 = np.cos(np.linspace(0, 6*np.pi, 291))                                      #1D  (291,)              Design matrix 2

print(prediction_accuracy.shape)
print(predicted_response.shape)
print(actual_response.shape)

In [ ]:
subject = 'ANfs'
xfmname = 'pcan-bbr'

vols = {}
vols['R2'] = cortex.Volume(prediction_accuracy.transpose(2,1,0), subject, xfmname)
vols['predicted'] = cortex.Volume(predicted_response.transpose(3,2,1,0), subject, xfmname)
vols['actual'] = cortex.Volume(actual_response.transpose(3,2,1,0), subject, xfmname)
vols['stimulus-1'] = design_matrix_1
vols['stimulus-2'] = design_matrix_2

cortex.webshow(vols, port=8899, open_browser=False)

In [2]:
def rgb_from_proj(proj3, chan_std):
    """(3,V) scaled PC coords -> (3,V) RGB in [0,1] (Huth clip/scale)."""
    return np.clip(proj3 / chan_std, -3, 3) / 3 / 2 + 0.5

def RGB2VertexRGB(data, alpha, subject):
    """RGB (V,3)/(3,V) -> cortex.VertexRGB (NaN -> transparent)."""
    data = np.asarray(data)
    rgb = data.T if data.shape[0] == 3 else data        # (V, 3)
    rgb = np.clip(np.nan_to_num(rgb), 0, 1)
    finite = np.ones(rgb.shape[0], bool)
    a = finite.astype("float32") if alpha is None else np.asarray(alpha, "float32")
    chans = [(rgb[:, i] * 255).astype("uint8") for i in range(3)]
    return cortex.VertexRGB(*chans, subject, alpha=a)


In [3]:

vertices = {}
for frames in ['_movie_delta_frames', '_movie_indep_delta_frames']:
    subject = 'fsaverage'
    rgb_filename = f'/data/jiwoongpark/statedynamics/experiments/storylistening/results/{frames}/_render.npz'

    rgb_data = np.load(rgb_filename)
    rgb_proj, rgb_alpha, rgb_chan_std = rgb_data['PROJ'], rgb_data["a"], rgb_data["chan_std"]
    print(rgb_proj.shape) # (291, 3, 327684)

    T = 291
    rgb_t = np.clip(np.nan_to_num(rgb_from_proj(rgb_proj, rgb_chan_std)), 0, 1)  # (T, 3, V)
    chans_t = [(rgb_t[:, i, :] * 255).astype('uint8') for i in range(3)]             # each (T, V)
    alpha_t = np.broadcast_to(np.asarray(rgb_alpha, 'float32'), (T, rgb_alpha.size)).copy()                            # (T, V)

    vertex_movie = cortex.VertexRGB(*chans_t, 'fsaverage', alpha=alpha_t)
    vertices[frames] = vertex_movie
cortex.webshow(vertices, port=8900, open_browser=False)

(291, 3, 327684)
(291, 3, 327684)
Started server on port 8900


Remote session detected -- after forwarding the port, open: http://localhost:8900/mixer.html


<WebApp(Thread-4, started 132834285303488)>